In [5]:
import sys 
from pathlib import Path
# repo root 
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd 
import numpy as np 


%load_ext autoreload
%autoreload 2
RAW_PATH = Path("../data/raw/kddcup.csv")
NAMES_PATH = Path("../data/raw/kddcup.names.txt")
Path("../src/__init__.py").touch()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
#peek or read the csv

from IPython.display import display
df_peek = pd.read_csv(RAW_PATH, header=None, nrows=3)
display(df_peek)


,0,1,2,3,4,5,6,7,8,9,...,32,33,34,35,36,37,38,39,40,41
0,0,tcp,http,SF,181,5450,0,0,0,0,...,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0,normal.
1,0,tcp,http,SF,239,486,0,0,0,0,...,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0,normal.
2,0,tcp,http,SF,235,1337,0,0,0,0,...,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0,normal.


In [ ]:
# 2. Load raw dataset via src.data and validate schema via src.validate
from src.data import load_raw
from src.validate import validate_raw

df = load_raw()
validate_raw(df)

# Test assertion
assert df.shape == (494021, 42), f'Expected (494021, 42), got {df.shape}'
print(f'Shape: {df.shape}')


In [ ]:
object_cols = df.select_dtypes(include=['object']).columns.tolist()
distinct_labels = df['label'].nunique()
ends_with_dot = df['label'].str.endswith('.').any()


print(f'Object columns: {object_cols}')
print(f'Distinct labels: {distinct_labels}')
print(f'Any end with ".": {ends_with_dot}')


In [16]:
attack_counts = df['label'].value_counts()
display(attack_counts.head(10))

label
smurf.          280790
neptune.        107201
normal.          97278
back.             2203
satan.            1589
ipsweep.          1247
portsweep.        1040
warezclient.      1020
teardrop.          979
pod.               264
Name: count, dtype: int64

In [17]:
num_duplicates = df.duplicated().sum()
total_rows = len(df)
dup_share = num_duplicates / total_rows

print(f"Exact duplicates: {num_duplicates:,}")
print(f"Share of file: {dup_share:.2%}")

# Duplicate rate per attack (duplicate copies / total for that label)
dup_per_attack = (df.duplicated().groupby(df["label"]).sum() / df["label"].value_counts()).sort_values(ascending=False)
display(dup_per_attack.head(5))


Exact duplicates: 348,435
Share of file: 70.53%


label
smurf.      280335
neptune.     71652
normal.      11682
back.         1365
satan.         877
dtype: int64

In [18]:
has_nan = df.isna().sum().sum()
numeric_df = df.select_dtypes(include=[np.number])
has_inf = np.isinf(numeric_df).sum().sum()

print(f"NaN count: {has_nan}")
print(f"Inf count: {has_inf}")

NaN count: 0
Inf count: 0


In [19]:
constant_cols = [col for col in df.columns if df[col].nunique() == 1]
print(f"Constant columns: {constant_cols}")

Constant columns: ['num_outbound_cmds', 'is_host_login']


In [21]:
rate_cols = [col for col in df.columns if col.endswith('_rate')]
rate_min = df[rate_cols].min().min()
rate_max = df[rate_cols].max().max()


numeric_df = df.select_dtypes(include=[np.number])
has_negative = (numeric_df < 0).sum().sum()

su_vals = sorted(df['su_attempted'].unique())

print(f"Rate columns range: [{rate_min}, {rate_max}]")
print(f"Negative values count: {has_negative}")
print(f"`su_attempted` unique values: {su_vals}")

Rate columns range: [0.0, 1.0]
Negative values count: 0
`su_attempted` unique values: [np.int64(0), np.int64(1), np.int64(2)]


In [22]:
protocol_cats = df['protocol_type'].nunique()
service_cats = df['service'].nunique()
flag_cats = df['flag'].nunique()

print(f"protocol_type categories: {protocol_cats}")
print(f"service categories: {service_cats}")
print(f"flag categories: {flag_cats}")

protocol_type categories: 3
service categories: 66
flag categories: 11


In [ ]:
# 10. If a model always predicted the most common label, what accuracy?
most_common_label = df['label'].value_counts().idxmax()
most_common_count = df['label'].value_counts().max()
accuracy = most_common_count / len(df)

print(f"Most common label: {most_common_label}")
print(f"Accuracy if always predicting '{most_common_label}': {accuracy:.2%}")

In [24]:
attack_types_path = Path("../data/raw/training_attack_types.txt")
assert attack_types_path.exists(), (
    f"{attack_types_path} not found. Download it from the KDD Cup 1999 data "
    f"page into data/raw/ before continuing."
)

FAMILY_MAP = {
    "dos": "DoS",
    "probe": "Probe",
    "r2l": "R2L",
    "u2r": "U2R",
}

attack_to_family = {}
with open(attack_types_path) as f:
    for line in f:
        parts = line.split()
        if len(parts) == 2:
            attack, family = parts
            attack_to_family[attack] = FAMILY_MAP.get(family.lower(), family)

attack_to_family["normal"] = "Normal"   # 'normal' is not listed in the attacks file

print(f"Mappings built from file: {len(attack_to_family)}")
print(f"Families present: {sorted(set(attack_to_family.values()))}")
assert sorted(set(attack_to_family.values())) == ["DoS", "Normal", "Probe", "R2L", "U2R"], \
    "Unexpected family names in training_attack_types.txt"
print("Built mapping from file ")

from src.config import ATTACK_TO_FAMILY
assert attack_to_family == ATTACK_TO_FAMILY, "attack_to_family does not match config.ATTACK_TO_FAMILY"
print("Mapping in config matches training_attack_types.txt exactly!")


Mappings built from file: 23
Families present: ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']
Built mapping from file 


In [ ]:
# Phase 2: Test the validator with deliberately broken inputs
# 1. Inject NaN
df_nan = df.copy()
df_nan.iloc[0, 0] = np.nan
try:
    validate_raw(df_nan)
except AssertionError as e:
    print(f"[PASS] Caught NaN: {e}")
else:
    raise RuntimeError("Validator did NOT catch NaN!")

# 2. Rename a column
df_rename = df.copy()
df_rename = df_rename.rename(columns={df.columns[0]: "wrong_column_name"})
try:
    validate_raw(df_rename)
except AssertionError as e:
    print(f"[PASS] Caught column mismatch: {str(e)[:50]}...")
else:
    raise RuntimeError("Validator did NOT catch column mismatch!")

# 3. Add an unknown label
df_unknown = df.copy()
df_unknown.iloc[0, df.columns.get_loc("label")] = "unknown_attack_type"
try:
    validate_raw(df_unknown)
except AssertionError as e:
    print(f"[PASS] Caught unknown label: {e}")
else:
    raise RuntimeError("Validator did NOT catch unknown label!")

# 4. Add a trailing dot on a known attack label to isolate the dot check
df_dot = df.copy()
df_dot.iloc[0, df.columns.get_loc("label")] = "normal."
try:
    validate_raw(df_dot)
except AssertionError as e:
    assert "ending with dot" in str(e), f"Expected dot check to fire, got: {e}"
    print(f"[PASS] Caught trailing dot: {e}")
else:
    raise RuntimeError("Validator did NOT catch trailing dot!")

print("All 4 broken input tests caught as expected!")
